In [ ]:
import os
import numpy as np
import scanpy as sc
import scvelo as scv
import matplotlib.pyplot as plt

# Step 2.1: Initialize global high-definition rendering parameters
scv.settings.set_figure_params('scanpy', dpi_save=400, frameon=False)
plt.rcParams['pdf.fonttype'] = 42      # Retains scalable typography strings inside vector platforms
plt.rcParams['svg.fonttype'] = 'none'  # Disables flattening text parameters into non-editable shapes

# Step 2.2: Establish spatial input/output directories
data_path = "/mnt/d/.Integrated_Final.h5ad"
save_path = "/mnt/d/.results/scV_plt"
os.makedirs(save_path, exist_ok=True)

print("Synchronizing final structural dynamic model matrix object...")
adata = sc.read_h5ad(data_path)

# Step 2.3: Data sanitization and coordinate overflow checks
print("Validating embedded trajectory spatial matrices...")
if 'velocity_umap' in adata.obsm:
    adata.obsm['velocity_umap'] = np.nan_to_num(adata.obsm['velocity_umap'], nan=0.0, posinf=0.0, neginf=0.0)
if 'X_umap' in adata.obsm:
    adata.obsm['X_umap'] = np.nan_to_num(adata.obsm['X_umap'], nan=0.0, posinf=0.0, neginf=0.0)

# Step 2.4: Axis range calculation for exact geometric matching
x_min, x_max = adata.obsm['X_umap'][:, 0].min(), adata.obsm['X_umap'][:, 0].max()
y_min, y_max = adata.obsm['X_umap'][:, 1].min(), adata.obsm['X_umap'][:, 1].max()

# Clip layout margins using a standard coordinate scale buffer
padding = 1.0
shared_xlim = [x_min - padding, x_max + padding]
shared_ylim = [y_min - padding, y_max + padding]

# Step 2.5: Generating side-by-side comparative dynamic line plots
print("Rendering group-stratified side-by-side lineage dynamic stream graphs...")
fig, axes = plt.subplots(1, 2, figsize=(16, 7)) 

experimental_groups = ['MOCK', 'ZIKV']

for idx, group_id in enumerate(experimental_groups):
    # Slice vector arrays isolated by current group attribute criteria
    subset_adata = adata[adata.obs['Group'] == group_id]
    
    # Render underlying velocity embedding trajectories to matching subplots
    scv.pl.velocity_embedding_stream(
        subset_adata,
        basis='umap',
        color='cell_type',
        ax=axes[idx],
        show=False,
        title=f'{group_id} Experimental Cohort',
        xlim=shared_xlim,
        ylim=shared_ylim,
        # Restrict legend instantiation exclusively to the right margin to prevent display compression
        legend_loc='none' if idx == 0 else 'right margin'
    )

plt.tight_layout()

# Step 2.6: High-DPI structural serialization export
output_file_path = os.path.join(save_path, 'velocity_umap_comparison.pdf')
try:
    plt.savefig(
        output_file_path, 
        format='pdf', 
        bbox_inches='tight', 
        transparent=True, 
        dpi=400
    )
    print(f"[SUCCESS] Comparative dynamic vector path committed safely to: {output_file_path}")
except Exception as e:
    print(f"[ERROR] PDF generation failed due to structure constraints: {e}")
    # Low-overhead fall-through save sequence
    raster_fallback = output_file_path.replace('.pdf', '.png')
    plt.savefig(raster_fallback, bbox_inches='tight', dpi=300)
    print(f"[SUCCESS] Fallback high-density raster chart saved to: {raster_fallback}")

plt.close()